# 01 - Data Preparation & Exploratory Data Analysis

## Objetivo
En este notebook se realiza:

- Extracción de datos desde la fuente oficial (MIDAGRI)
- Limpieza y transformación de la serie temporal
- Validación de la calidad de datos
- Análisis exploratorio de la serie
- Identificación de tendencia, estacionalidad y patrones
- Análisis de autocorrelación (ACF y PACF)

Este paso es fundamental para entender la estructura de la serie antes del modelado.

In [1]:
## Fuente # Sofia estuvo aquí
# Portal SIEA - MIDAGRI:
# https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo

In [25]:
# AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA

In [26]:
# :P

In [24]:
# HOLA HOLA HOLA

In [2]:
# Librerías base
import re
import io
import os
import time
import requests
import warnings
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

# Manejo de datos
import numpy as np
import pandas as pd

# Parsing HTML
from bs4 import BeautifulSoup

# Lectura de PDF
import pdfplumber

# Visualización
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
SITE_URL = "https://siea.midagri.gob.pe"
BASE_URL = "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo"

YEAR_URLS = [
    "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025",
    "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026"
]

### Extraer meses

In [4]:
# Aquí guardaremos todos los meses
all_month_links = []

# Mozilla/5. es un User-Agent que ayuda a que la web no bloquee la solicitud, para q no nos salga error 403
headers = {"User-Agent": "Mozilla/5.0"}

# Recorremos cada año (2025 y 2026)
for year_url in YEAR_URLS:
        
    #  Pedimos el HTML de la página
    response = requests.get(year_url, headers=headers)
    
    #  Convertimos a formato BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser") #beautifulSoup hace q el formato html se pueda leer en python
    
    #  Sacamos todos los links <a href="...">
    links = []
    
    for a in soup.find_all("a"):
        if a.get("href") is not None:
            
            href = a.get("href")
            
            # Convertimos a link completo
            full_link = urljoin(SITE_URL, href)
            
            links.append(full_link)
    
    # Quitamos duplicados
    links = list(set(links))
    
    #  Filtramos solo los meses
    month_links = []
    
    for link in links:
        
        if BASE_URL in link:          # pertenece a la sección de huevo
            if link != year_url:      # no es la página del año
                if "download=" not in link:  # no es descarga
                    month_links.append(link)
    
    
    # Guardamos
    all_month_links.extend(month_links)

# Quitamos duplicados finales
all_month_links = list(set(all_month_links))

In [5]:
df_months = pd.DataFrame({"month_url": all_month_links})

df_months #hay 8 links

,month_url
0,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...
5,https://siea.midagri.gob.pe/portal/publicacion...
6,https://siea.midagri.gob.pe/portal/publicacion...
7,https://siea.midagri.gob.pe/portal/publicacion...
8,https://siea.midagri.gob.pe/portal/publicacion...


In [6]:
# Aquí guardaremos todos los registros de todos los meses
all_daily_records = []

# Lista de nombres de meses para detectar fechas en texto
meses = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "setiembre", "septiembre",
    "octubre", "noviembre", "diciembre"
]

In [7]:
# recorrer mes a mes

for month_url in all_month_links:
    
    print(month_url)
    
    # 1. Descargar el HTML del mes
    response = requests.get(month_url, headers=headers)
    
    # 2. Convertir a BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 3. Revisar todos los bloques div(organizan) de la página
    for div in soup.find_all("div"):
        
        texto = div.get_text(" ", strip=True) #extrae el texto dentro de cada div
        
        # 4. Nos quedamos solo con bloques que parecen boletines
        # porque contienen un mes en el texto y además la palabra DESCARGAR
        if any(mes in texto.lower() for mes in meses) and "descargar" in texto.lower():
            
            # 5. Buscar links dentro de ese bloque
            links_locales = []
            
            for a in div.find_all("a"):
                href = a.get("href")
                
                if href is not None:
                    link_completo = urljoin(SITE_URL, href)
                    links_locales.append(link_completo)
            
            # 6. Buscar el link de descarga del PDF
            download_url = None
            
            for link in links_locales:
                if "download=" in link:
                    download_url = link
                    break
            
            # 7. Guardar el registro
            all_daily_records.append({
                "month_url": month_url,
                "texto_boletin": texto,
                "download_url": download_url
            })

https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/536-huevo-marzo
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/470-huevo-octubre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/487-huevo-noviembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/549-huevo-abril
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/496-huevo-diciembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/469-huevo-setiembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/499-huevo-e

In [8]:
#convertimos a dataframe
df_daily_catalog = pd.DataFrame(all_daily_records)

In [9]:
df_daily_catalog

,month_url,texto_boletin,download_url
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...
...,...,...,...
139,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
140,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
141,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
142,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...


In [10]:
print("Total de registros encontrados:", len(df_daily_catalog))
print("Registros con link de descarga:", df_daily_catalog["download_url"].notna().sum())

Total de registros encontrados: 144
Registros con link de descarga: 144


In [11]:
#limpiamos duplicados
df_daily_catalog = df_daily_catalog.drop_duplicates().reset_index(drop=True)

print("Total después de quitar duplicados:", len(df_daily_catalog))

Total después de quitar duplicados: 104


In [12]:
df_daily_catalog

,month_url,texto_boletin,download_url
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
100,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
101,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
102,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...


Ya tenemos todos los pdfs, ahora normalizaremos los nombres

In [13]:
import re


In [14]:
fechas_extraidas = []

for texto in df_daily_catalog["texto_boletin"]:
    
    texto = str(texto).lower()
    
    patron = r"(\d{1,2}\s+(?:enero|febrero|marzo|abril|mayo|junio|julio|agosto|setiembre|septiembre|octubre|noviembre|diciembre)\s+\d{4})"
    
    match = re.search(patron, texto)
    
    if match:
        fechas_extraidas.append(match.group(1))
    else:
        fechas_extraidas.append(None)

df_daily_catalog["fecha_texto"] = fechas_extraidas

In [15]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
1,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
2,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
3,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026
4,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026
100,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026
101,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026
102,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026


In [16]:
#eliminamos filas sin fecha o sin pdf
df_daily_catalog = df_daily_catalog[
    df_daily_catalog["fecha_texto"].notna() &
    df_daily_catalog["download_url"].notna()
].reset_index(drop=True)

print("Total final de boletines válidos:", len(df_daily_catalog))

Total final de boletines válidos: 104


In [17]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
1,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
2,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
3,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026
4,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026
100,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026
101,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026
102,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026


DESARGAR LOS PDFS

In [18]:
import os
os.getcwd()
os.listdir()

['00_leer_pdfs.ipynb',
 '01_extraer_informacion.ipynb',
 '02_preprocesar_data_final.ipynb',
 '03_analisis_serie_de_tiempo.ipynb',
 'datos_tabla1.csv',
 'datos_tabla2.csv',
 'datos_tabla3.csv',
 'pdfs_procesados.txt',
 'pdfs_procesados_tabla2.txt',
 'pdfs_procesados_tabla3.txt']

In [19]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
1,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
2,https://siea.midagri.gob.pe/portal/publicacion...,2026 Marzo 2026 31 marzo 2026 DESCARGAR VISUAL...,https://siea.midagri.gob.pe/portal/publicacion...,26 marzo 2026
3,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 marzo 2026
4,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 marzo 2026
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 febrero 2026
100,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 febrero 2026
101,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,18 febrero 2026
102,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,17 febrero 2026


In [20]:
import os
pdf_folder = "../pdfs"

DESCARGAMOS solo lo nuevo para evitar descargar todos los pdfs cada que abrimos el código

In [21]:
import requests
import os

for i, row in df_daily_catalog.iterrows():
    url = row["download_url"]
    fecha = row["fecha_texto"]
    
    fecha_clean = fecha.replace(" ", "_").replace("/", "-")
    file_path = f"{pdf_folder}/{fecha_clean}.pdf"
    
    if os.path.exists(file_path):
        print(f"Saltando {fecha} (ya existe)")
        continue
    
    print(f"Descargando {fecha}")
    try:
        response = requests.get(url)
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f" Descargado")
    except Exception as e:
        print(f"Error: {e}")

Saltando 26 marzo 2026 (ya existe)
Saltando 26 marzo 2026 (ya existe)
Saltando 26 marzo 2026 (ya existe)
Saltando 31 marzo 2026 (ya existe)
Saltando 30 marzo 2026 (ya existe)
Saltando 27 marzo 2026 (ya existe)
Saltando 26 marzo 2026 (ya existe)
Saltando 25 marzo 2026 (ya existe)
Saltando 24 marzo 2026 (ya existe)
Saltando 23 marzo 2026 (ya existe)
Saltando 20 marzo 2026 (ya existe)
Saltando 19 marzo 2026 (ya existe)
Saltando 18 marzo 2026 (ya existe)
Saltando 25 octubre 2025 (ya existe)
Saltando 25 octubre 2025 (ya existe)
Saltando 25 octubre 2025 (ya existe)
Saltando 31 octubre 2025 (ya existe)
Saltando 30 octubre 2025 (ya existe)
Saltando 29 octubre 2025 (ya existe)
Saltando 28 octubre 2025 (ya existe)
Saltando 27 octubre 2025 (ya existe)
Saltando 24 octubre 2025 (ya existe)
Saltando 23 octubre 2025 (ya existe)
Saltando 22 octubre 2025 (ya existe)
Saltando 21 octubre 2025 (ya existe)
Saltando 20 octubre 2025 (ya existe)
Saltando 25 noviembre 2025 (ya existe)
Saltando 25 noviembre 202

lista de los archivos pdfs

In [22]:
pdf_folder = "../pdfs"

# Lista de archivos PDF
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]
pdf_files

['09_abril_2026.pdf',
 '10_abril_2026.pdf',
 '12_diciembre_2025.pdf',
 '13_abril_2026.pdf',
 '13_febrero_2026.pdf',
 '14_abril_2026.pdf',
 '15_abril_2026.pdf',
 '15_diciembre_2025.pdf',
 '16_abril_2026.pdf',
 '16_diciembre_2025.pdf',
 '16_enero_2026.pdf',
 '16_setiembre_2025.pdf',
 '17_abril_2026.pdf',
 '17_diciembre_2025.pdf',
 '17_febrero_2026.pdf',
 '17_noviembre_2025.pdf',
 '17_setiembre_2025.pdf',
 '18_diciembre_2025.pdf',
 '18_febrero_2026.pdf',
 '18_marzo_2026.pdf',
 '18_noviembre_2025.pdf',
 '18_setiembre_2025.pdf',
 '19_diciembre_2025.pdf',
 '19_enero_2026.pdf',
 '19_febrero_2026.pdf',
 '19_marzo_2026.pdf',
 '19_noviembre_2025.pdf',
 '19_setiembre_2025.pdf',
 '20_abril_2026.pdf',
 '20_enero_2026.pdf',
 '20_febrero_2026.pdf',
 '20_marzo_2026.pdf',
 '20_noviembre_2025.pdf',
 '20_octubre_2025.pdf',
 '21_abril_2026.pdf',
 '21_enero_2026.pdf',
 '21_noviembre_2025.pdf',
 '21_octubre_2025.pdf',
 '22_abril_2026.pdf',
 '22_diciembre_2025.pdf',
 '22_enero_2026.pdf',
 '22_octubre_2025.pd